# 06 · PyTorch 训练循环骨架（MNIST MLP）

> **学习目标**：用不到 100 行 PyTorch 训一个 MLP，MNIST 准确率 95%+。把 `DataLoader → forward → loss → backward → optimizer.step` 这个 5 步骨架**刻进肌肉记忆**。
>
> **预备**：04 已过。本机 `rag` env 有 `torch 2.8 + cu126`，CUDA 可用。
>
> **为什么重要**：所有后续训练（LoRA / SFT / DPO）都是这个骨架的变形。骨架你能背了，看任何训练脚本都不会发懵。

**第一次跑时**：torchvision 会从外网下 MNIST（约 11 MB）。如果下不动，本 notebook 末尾有「离线下载步骤」。

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 1. 数据：DataLoader

**心智模型**：
- `Dataset` 一次返回 1 个样本（`__getitem__`）
- `DataLoader` 包一层，做 batching / shuffle / 并行预读

**关键超参**：`batch_size`、`shuffle=True`（训练）/ `False`（评估）、`num_workers`（多进程读数据）。

In [ ]:
# transform: PIL Image -> Tensor(C=1, H=28, W=28), 像素 [0,1]，再标准化
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),    # MNIST 全集均值/标准差，sklearn 风格
])

DATA_DIR = './_mnist_data'
train_set = datasets.MNIST(DATA_DIR, train=True,  download=True, transform=transform)
test_set  = datasets.MNIST(DATA_DIR, train=False, download=True, transform=transform)

print('训练样本数:', len(train_set))
print('测试样本数:', len(test_set))
print('单样本 shape:', train_set[0][0].shape, '  label:', train_set[0][1])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=0)

In [ ]:
# 看 9 张图验证数据正常
fig, axes = plt.subplots(3, 3, figsize=(5, 5))
for ax, (img, label) in zip(axes.flat, train_set):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'label={label}')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2. 模型：MLP（不许用 CNN）

**结构**：`784 (= 28×28) → 256 → 128 → 10`，中间用 ReLU + Dropout。

**为什么这么定**：足够过 95% 的最小结构。MLP 在 MNIST 上能上 98%，但更大就是 bonus。

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=784, h1=256, h2=128, n_classes=10, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                       # (B,1,28,28) -> (B,784)
            nn.Linear(in_dim, h1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h1, h2),     nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h2, n_classes),           # 输出 logits，不要在这里 softmax
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f'\n参数量: {n_params:,} ({n_params/1e6:.2f} M)')

## 3. 训练循环：5 步骨架

```python
for batch in loader:
    optimizer.zero_grad()             # 1. 清梯度
    pred = model(x)                   # 2. 前向
    loss = criterion(pred, y)         # 3. 算 loss
    loss.backward()                   # 4. 反向（计算梯度）
    optimizer.step()                  # 5. 更新参数
```

**别忘**：
- 模型有 dropout / batchnorm 时，**训练 `model.train()`、评估 `model.eval()`**，否则结果错
- 评估时用 `with torch.no_grad():` 关掉梯度计算，省显存提速度

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss    += loss.item() * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_count   += x.size(0)
    return total_loss / total_count, total_correct / total_count

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss    += loss.item() * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_count   += x.size(0)
    return total_loss / total_count, total_correct / total_count

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 5
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    tl, ta = train_one_epoch(model, train_loader, optimizer, criterion)
    vl, va = evaluate(model, test_loader, criterion)
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['test_loss'].append(vl);  history['test_acc'].append(va)
    print(f'Epoch {epoch}: train loss {tl:.4f}  acc {ta*100:5.2f}%  ||  test loss {vl:.4f}  acc {va*100:5.2f}%')

print('\n最终测试准确率:', round(history["test_acc"][-1] * 100, 2), '%')
assert history['test_acc'][-1] > 0.95, '准确率没到 95%，检查 lr / epoch / dropout'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history['train_loss'], label='train'); axes[0].plot(history['test_loss'], label='test')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history['train_acc'],  label='train'); axes[1].plot(history['test_acc'],  label='test')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

## 4. 看错例 —— 比看正确率有用 10 倍

**调试黄金法则**：模型烂的时候不要瞎调超参，**先看它在哪些样本上错**。错例的模式告诉你下一步该往哪改。

In [ ]:
model.eval()
wrong = []                                # (img, true, pred)
with torch.no_grad():
    for x, y in test_loader:              # x, y 在 cpu
        x_dev, y_dev = x.to(device), y.to(device)
        pred = model(x_dev).argmax(1)
        mask = (pred != y_dev).cpu()       # 把 mask 拉回 cpu，才能去 index cpu 上的 x/y
        for img, t, p in zip(x[mask], y[mask], pred.cpu()[mask]):
            wrong.append((img, int(t), int(p)))
        if len(wrong) >= 9:
            break

fig, axes = plt.subplots(3, 3, figsize=(5, 5))
for ax, (img, t, p) in zip(axes.flat, wrong[:9]):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'真={t} 预测={p}', color='red')
    ax.axis('off')
plt.tight_layout(); plt.show()
print('观察：错的样本是不是确实「写得潦草」？如果连人都难分辨，那是数据极限，不是模型问题。')

In [ ]:
# 混淆矩阵：哪类被错认成哪类
from collections import Counter
import numpy as np

cm = np.zeros((10, 10), dtype=int)
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        pred = model(x.to(device)).argmax(1).cpu().numpy()
        for t, p in zip(y.numpy(), pred):
            cm[t, p] += 1

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
for i in range(10):
    for j in range(10):
        c = cm[i, j]
        if c:
            plt.text(j, i, c, ha='center', va='center',
                     color='white' if c > cm.max()/2 else 'black', fontsize=8)
plt.xlabel('predicted'); plt.ylabel('true')
plt.title('Confusion Matrix')
plt.colorbar(); plt.show()

# 哪两类最容易搞混
off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f'最常见错误：真值 {i} → 预测 {j}（{off[i,j]} 次）')

## 深入思考

1. **`optimizer.zero_grad()` 不调会怎样？**
   - 梯度会**累加**到上一步，跑两步就开始炸。除非你刻意做梯度累积。
2. **`model.eval()` 关掉的是什么？**
   - Dropout 不再丢神经元；BatchNorm 用统计量而不是 batch 当前均值方差。
3. **训练 acc 高、测试 acc 低，说明什么？**
   - 过拟合。手段：加 Dropout / 加 weight decay / 减小模型 / 多数据 / 早停。
4. **Adam vs SGD 该选谁？**
   - 入门 / 实验：Adam（开箱即用）。极致性能 / 大规模训练：SGD + momentum（更难调但常常上限更高）。
5. **训不下去的「5 步排查」**：
   1. 学习率太大/太小？（试 [1e-2, 1e-3, 1e-4]）
   2. 数据有没有标准化？
   3. label 对不对（看几条 `train_set[i]` 验证）
   4. 是不是 `train()/eval()` 切换错了
   5. 损失函数和输出维度匹配吗（10 类用 CE，输出 10 维 logits 不要 softmax）

改一改：把 lr 改成 `1e-1`，看 loss 是不是炸了。把 dropout 改成 0.5，看测试 acc。

## 自检 ✅

- [ ] 不查文档 5 分钟内写完训练循环 5 步骨架。
- [ ] 解释 `model.train() / eval()` 切换的作用。
- [ ] 解释为什么用 CE 时输出层不应该 softmax。
- [ ] 给一个 MNIST acc 卡在 80% 的脚本，5 分钟内列出 5 个排查方向。
- [ ] 复述上一个 cell 的「调不下去 5 步」。

## 离线下载 MNIST（备用）

如果首次 download 失败：手动下载 4 个文件放到 `./_mnist_data/MNIST/raw/`：
- `train-images-idx3-ubyte.gz` / `train-labels-idx1-ubyte.gz` / `t10k-images-idx3-ubyte.gz` / `t10k-labels-idx1-ubyte.gz`
- 镜像：`https://ossci-datasets.s3.amazonaws.com/mnist/`

## 下一步

→ [`07_self_attention.ipynb`](07_self_attention.ipynb)